# PSD + LDA Seizure Prediction
Cross-patient benchmarking on CHB-MIT · 20 seeds · mean ± std

**Memory-efficient version**: extracts PSD features per-patient before concatenation.

In [ ]:
from pathlib import Path
import sys

def find_repo_root(start):
    for path in [start, *start.parents]:
        if (path / 'src').exists() and (path / 'README.md').exists():
            return path
    return start

ROOT = find_repo_root(Path.cwd())
sys.path.insert(0, str(ROOT / 'src'))


In [ ]:
import os
import json
import numpy as np
from scipy.signal import welch
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import roc_auc_score

import sys
sys.path.insert(0, str(ROOT / 'src'))
from data_utils import make_patient_splits, SEEDS, DATA_DIR
from eval_utils import find_youden_threshold, full_evaluate

FS      = 256
WIN     = 20 * FS
OUT_DIR = r'D:\seizure_results\psd_lda_leaky'

BANDS = [
    (0.5,  4),   # delta
    (4,    8),   # theta
    (8,   13),   # alpha
    (13,  30),   # beta
    (30,  40),   # gamma
]

os.makedirs(OUT_DIR, exist_ok=True)
print(f'Output dir: {OUT_DIR}')
print(f'Seeds     : {len(SEEDS)}')
print(f'Features  : 18 channels x {len(BANDS)} bands = {18 * len(BANDS)} dims')

In [ ]:
def extract_psd_features(X):
    """
    Extract PSD band-power features from raw EEG windows.
    
    Parameters
    ----------
    X : ndarray, shape (n_windows, 18, 5120)
        Raw EEG windows.
    
    Returns
    -------
    features : ndarray, shape (n_windows, 90)
        Mean PSD in each of the 5 frequency bands for each of 18 channels.
    """
    freqs, pxx = welch(X, fs=FS, axis=-1, nperseg=512)
    band_feats = []
    for band_lo, band_hi in BANDS:
        mask = (freqs >= band_lo) & (freqs <= band_hi)
        band_power = pxx[:, :, mask].mean(axis=-1)  # (n, 18)
        band_feats.append(band_power)
    return np.concatenate(band_feats, axis=1)  # (n, 90)

In [ ]:
def load_psd_features(patient_list, data_dir):
    X_list, y_list = [], []
    for pt in patient_list:
        X_raw = np.load(os.path.join(data_dir, f"{pt}_X.npy"))
        y     = np.load(os.path.join(data_dir, f"{pt}_y.npy"))
        
        feat = extract_psd_features(X_raw)
        X_list.append(feat)
        y_list.append(y)
        
        del X_raw
    return np.concatenate(X_list, axis=0), np.concatenate(y_list, axis=0)

In [ ]:
# evaluate() removed - using eval_utils.full_evaluate instead
print('Using eval_utils.full_evaluate for all metrics.')

In [ ]:
def run_seed(seed):
    print(f"\n{'=' * 60}")
    print(f"  Seed {seed}")
    print(f"{'=' * 60}")

    # --- leaky: pool all patients, random stratified k-fold ---
    from sklearn.model_selection import StratifiedKFold
    import numpy as np

    VALID_PATIENTS = [
        'chb01','chb02','chb03','chb04','chb05','chb06','chb07','chb08',
        'chb09','chb10','chb11','chb12','chb13','chb14','chb15','chb16',
        'chb17','chb18','chb19','chb20','chb21','chb22','chb23',
    ]

    print('  Loading PSD features for all patients...')
    X_all, y_all = load_psd_features(VALID_PATIENTS, DATA_DIR)
    print(f"  total={len(y_all)}  pre={y_all.sum()}  inter={(y_all==0).sum()}")

    skf   = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    folds = list(skf.split(X_all, y_all))
    _, test_idx = folds[0]
    _, val_idx  = folds[1]
    train_idx   = np.concatenate([tr for tr, _ in folds[2:]])

    X_train, y_train = X_all[train_idx], y_all[train_idx]
    X_val,   y_val   = X_all[val_idx],   y_all[val_idx]
    X_test,  y_test  = X_all[test_idx],  y_all[test_idx]
    del X_all, y_all

    print(f"  split -> train={len(y_train)}  val={len(y_val)}  test={len(y_test)}")

    print('  Training LDA...')
    lda = LinearDiscriminantAnalysis(solver='svd', priors=[0.5, 0.5])
    lda.fit(X_train, y_train)

    val_prob  = lda.predict_proba(X_val)[:, 1]
    val_auc   = roc_auc_score(y_val, val_prob)
    threshold = find_youden_threshold(y_val, val_prob)
    print(f"  Val  AUC {val_auc:.4f} | Youden threshold: {threshold:.4f}")

    test_prob = lda.predict_proba(X_test)[:, 1]
    m = full_evaluate(y_test, test_prob, threshold, stride_s=300)

    print(f"  [Seed {seed}] TEST  AUC {m['auc']:.4f} | "
          f"Sen {m['sensitivity']:.4f} | Spe {m['specificity']:.4f} | "
          f"Prec {m['precision']:.4f} | F1 {m['f1']:.4f} | "
          f"FAR {m['far']:.3f}/h | "
          f"EvtSen {m['event_sensitivity']:.3f} ({m['n_events']} events)")

    return {
        'seed':               seed,
        'test_auc':           m['auc'],
        'test_sen':           m['sensitivity'],
        'test_spe':           m['specificity'],
        'test_precision':     m['precision'],
        'test_f1':            m['f1'],
        'far':                m['far'],
        'event_sensitivity':  m['event_sensitivity'],
        'n_events':           m['n_events'],
        'threshold':          float(threshold),
        'best_val_auc':       float(val_auc),
    }


In [ ]:
all_results = []
for s in [42]:
    result = run_seed(s)
    all_results.append(result)

results_path = os.path.join(OUT_DIR, 'results.json')
with open(results_path, 'w') as f:
    json.dump(
        [{k: (int(v) if k in ('seed', 'n_events') else float(v))
          for k, v in r.items()}
         for r in all_results],
        f, indent=2,
    )
print(f"\nResults saved to {results_path}")

In [ ]:
metrics = ['test_auc', 'test_sen', 'test_spe', 'test_precision',
           'test_f1', 'far', 'event_sensitivity']
labels  = ['AUC', 'Sensitivity (win)', 'Specificity', 'Precision',
           'F1', 'FAR (/h)', 'Sensitivity (event)']

print(f"\nPSD+LDA  Cross-Patient Results (mean ± std, n={len(all_results)} seeds)")
print('-' * 55)
for m, l in zip(metrics, labels):
    vals = np.array([r[m] for r in all_results])
    print(f'  {l:<22s}: {vals.mean():.4f} ± {vals.std(ddof=1):.4f}')

In [ ]:
import timeit
import numpy as np

train_pts, val_pts, test_pts = make_patient_splits(42)
X_train, y_train = load_psd_features(train_pts, DATA_DIR)
X_test_s, y_test_s = load_psd_features(test_pts, DATA_DIR)

lda_bench = LinearDiscriminantAnalysis(solver='svd', priors=[0.5, 0.5])
lda_bench.fit(X_train, y_train)

single_sample = X_test_s[:1]   # shape (1, 90)

times = timeit.repeat(lambda: lda_bench.predict_proba(single_sample), repeat=5, number=200)
median_ms = np.median(times) / 200 * 1000
print(f"PSD+LDA single-sample inference latency: {median_ms:.2f} ms")